<a href="https://colab.research.google.com/github/yeshaa23/ZARA-AppsReview-SentimentAnalysis/blob/main/Scrapping_link_Pertamina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scrapping dan Preprocessing link artikel Pertamina
100 link

Install dan import Library

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install contractions openpyxl Sastrawi -q

In [3]:
import pandas as pd
import numpy as np
import re
import html
import contractions
import warnings
warnings.filterwarnings("ignore")

## UPLOAD FILE CSV
pakai excel file manual

In [4]:
df = pd.read_excel('/content/drive/MyDrive/ProjectA-PBA/artikel_manual_pertamina.xlsx')

In [5]:
print("Jumlah data awal:", len(df))
print("Kolom awal:", df.columns.tolist())
display(df.head())

Jumlah data awal: 50
Kolom awal: ['Case ID', 'Case', 'Newsroom', 'Tanggal', 'Judul', 'Konten', 'Link', 'Tag', 'Tone']


,Case ID,Case,Newsroom,Tanggal,Judul,Konten,Link,Tag,Tone
0,NaN,NaN,ylki.or.id,2016-06-08 00:00:00,Siaran Pers YLKI : Mendesak PT Pertamina untuk...,Kejadian kecurangan takaran pada SPBU yang ter...,https://ylki.or.id/siaran-pers-ylki-mendesak-p...,Hukum,Negatif
1,NaN,NaN,Kompas.com,6 Juli 2017,Pertamina Kembangkan Program Non-Tunai Pembeli...,"PONTIANAK, KOMPAS.com - PT Pertamina mensosial...",https://money.kompas.com/read/2017/07/06/14535...,BBM,Netral
2,NaN,NaN,Kompas.com,14 Desember 2017,"Pertamina Tak Masalahkan Nama Pertamini, tapi...","Nusa Dua, KompasOtomotif - Keberadaan kios-kio...",https://otomotif.kompas.com/read/2017/12/14/15...,Bisnis,Positif
3,NaN,NaN,Kompas.com,27 Juli 2017,Pertamina Patra Niaga Akan Bangun SPBU Skala U...,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga...",https://money.kompas.com/read/2017/07/27/17565...,Bisnis,Positif
4,NaN,NaN,Kompas.com,18 Maret 2026,"Gaji Manajer 'Engineering' Chevron Rp 51,25 Ju...","JAKARTA, KOMPAS.com - Situs pencari kerja, Job...",https://money.kompas.com/read/2016/03/18/14152...,Bisnis,Negatif


In [6]:
# rapikan nama kolom kalau ada spasi
df.columns = df.columns.str.strip()

# rename
rename_map = {
    'Judul': 'title',
    'Konten': 'content',
    'Tag': 'tag',
    'Tone': 'sentiment',
    'Tanggal': 'date',
    'Newsroom': 'source',
    'Link': 'url',
    'Case ID': 'case_id',
    'Case': 'case'
}
df = df.rename(columns=rename_map)

print("Kolom setelah rename:", df.columns.tolist())

Kolom setelah rename: ['case_id', 'case', 'source', 'date', 'title', 'content', 'url', 'tag', 'sentiment']


In [7]:
# pilih kolom
cols_to_keep = [col for col in ['case_id', 'case', 'source', 'date', 'title', 'content', 'url', 'tag', 'sentiment'] if col in df.columns]
df = df[cols_to_keep].copy()

## Cleaning data

In [8]:
# drop baris yang kontennya kosong
df = df.dropna(subset=['content']).copy()

# ubah ke string
for col in ['title', 'content', 'tag', 'sentiment', 'source', 'url']:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()

# buang isi content yang kosong setelah strip
df = df[df['content'].str.strip() != ''].copy()

print("Jumlah data setelah drop empty content:", len(df))

Jumlah data setelah drop empty content: 50


In [9]:
# hapus duplikat
print("Duplikat content sebelum hapus:", df['content'].duplicated().sum())

df = df.drop_duplicates(subset=['content']).copy()

if 'url' in df.columns:
    df = df.drop_duplicates(subset=['url']).copy()

print("Jumlah data setelah hapus duplikat:", len(df))

Duplikat content sebelum hapus: 0
Jumlah data setelah hapus duplikat: 50


## Function Preprocessing untuk sentiment

In [10]:
def sentiment_preprocessing(text: str) -> str:
    text = html.unescape(str(text))
    # normalize quotes
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r"[‘’]", "'", text)
    # collapse duplicate quotes
    text = re.sub(r'""', '"', text)
    text = re.sub(r"''", "'", text)
    # remove escaped newline/tab
    text = re.sub(r'\\[nrt]+', ' ', text)
    # remove url / mention
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    # contractions, mostly for English mixed text
    text = contractions.fix(text)
    # lowercase
    text = text.lower()
    # keep only useful punctuation
    text = re.sub(r"[^\w\s!?%']", " ", text)
    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [11]:
def ner_pos_preprocessing(text: str) -> str:
    text = html.unescape(str(text))

    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r"[‘’]", "'", text)

    text = re.sub(r'""', '"', text)
    text = re.sub(r"''", "'", text)

    text = re.sub(r'\\[nrt]+', ' ', text)

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    text = contractions.fix(text)

    text = re.sub(r'[ \t\r]+', ' ', text)
    text = re.sub(r' *\n *', '\n', text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

## Apply Preprocessing

In [12]:
df['content_clean_sentiment'] = df['content'].apply(sentiment_preprocessing)
df['content_clean_nerpos'] = df['content'].apply(ner_pos_preprocessing)

display(df[['content', 'content_clean_sentiment', 'content_clean_nerpos']].head())

,content,content_clean_sentiment,content_clean_nerpos
0,Kejadian kecurangan takaran pada SPBU yang ter...,kejadian kecurangan takaran pada spbu yang ter...,Kejadian kecurangan takaran pada SPBU yang ter...
1,"PONTIANAK, KOMPAS.com - PT Pertamina mensosial...",pontianak kompas com pt pertamina mensosialisa...,"PONTIANAK, KOMPAS.com - PT Pertamina mensosial..."
2,"Nusa Dua, KompasOtomotif - Keberadaan kios-kio...",nusa dua kompasotomotif keberadaan kios kios p...,"Nusa Dua, KompasOtomotif - Keberadaan kios-kio..."
3,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga...",jakarta kompas com pt pertamina patra niaga se...,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga..."
4,"JAKARTA, KOMPAS.com - Situs pencari kerja, Job...",jakarta kompas com situs pencari kerja jobplan...,"JAKARTA, KOMPAS.com - Situs pencari kerja, Job..."


In [13]:
df['content_length'] = df['content'].astype(str).apply(len)
df['clean_length'] = df['content_clean_sentiment'].astype(str).apply(len)

print("Jumlah data final:", len(df))
print("Artikel dengan clean text kosong:", (df['content_clean_sentiment'].str.strip() == '').sum())
print("Artikel dengan content < 200 char:", (df['content_length'] < 200).sum())

Jumlah data final: 50
Artikel dengan clean text kosong: 0
Artikel dengan content < 200 char: 0


## Simpan hasil

In [16]:
output_path = '/content/drive/MyDrive/ProjectA-PBA/artikel_manual_pertamina_preprocessed.csv'
df.to_csv(output_path, index=False)

print("File preprocessing disimpan di:", output_path)
print("Ukuran akhir data:", df.shape)
display(df.head())

File preprocessing disimpan di: /content/drive/MyDrive/ProjectA-PBA/artikel_manual_pertamina_preprocessed.csv
Ukuran akhir data: (50, 13)


,case_id,case,source,date,title,content,url,tag,sentiment,content_clean_sentiment,content_clean_nerpos,content_length,clean_length
0,NaN,NaN,ylki.or.id,2016-06-08 00:00:00,Siaran Pers YLKI : Mendesak PT Pertamina untuk...,Kejadian kecurangan takaran pada SPBU yang ter...,https://ylki.or.id/siaran-pers-ylki-mendesak-p...,Hukum,Negatif,kejadian kecurangan takaran pada spbu yang ter...,Kejadian kecurangan takaran pada SPBU yang ter...,1500,1465
1,NaN,NaN,Kompas.com,6 Juli 2017,Pertamina Kembangkan Program Non-Tunai Pembeli...,"PONTIANAK, KOMPAS.com - PT Pertamina mensosial...",https://money.kompas.com/read/2017/07/06/14535...,BBM,Netral,pontianak kompas com pt pertamina mensosialisa...,"PONTIANAK, KOMPAS.com - PT Pertamina mensosial...",3555,3459
2,NaN,NaN,Kompas.com,14 Desember 2017,"Pertamina Tak Masalahkan Nama Pertamini, tapi...","Nusa Dua, KompasOtomotif - Keberadaan kios-kio...",https://otomotif.kompas.com/read/2017/12/14/15...,Bisnis,Positif,nusa dua kompasotomotif keberadaan kios kios p...,"Nusa Dua, KompasOtomotif - Keberadaan kios-kio...",1610,1561
3,NaN,NaN,Kompas.com,27 Juli 2017,Pertamina Patra Niaga Akan Bangun SPBU Skala U...,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga...",https://money.kompas.com/read/2017/07/27/17565...,Bisnis,Positif,jakarta kompas com pt pertamina patra niaga se...,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga...",2004,1941
4,NaN,NaN,Kompas.com,18 Maret 2026,"Gaji Manajer 'Engineering' Chevron Rp 51,25 Ju...","JAKARTA, KOMPAS.com - Situs pencari kerja, Job...",https://money.kompas.com/read/2016/03/18/14152...,Bisnis,Negatif,jakarta kompas com situs pencari kerja jobplan...,"JAKARTA, KOMPAS.com - Situs pencari kerja, Job...",1721,1677
